# CogAttention — Stimulus-Driven Attention

**Track:** Attention — Stimulus-Driven Attention
**Benchmark:** CogAttention v1.0
**Tasks:** anomaly

---

## Methodology

Tests stimulus-driven attention through dual-task anomaly detection. Model must complete a primary counting task while detecting embedded anomalies. Based on Inattentional Blindness (Simons & Chabris, 1999).

### Cognitive Science Grounding

- **Inattentional blindness** (Simons & Chabris, 1999): failing to notice an unexpected stimulus when attention is engaged in a primary task. Our dual-task paradigm requires counting/summing while an anomaly is embedded in the passage.
- **Stimulus-driven / exogenous attention** (Posner, 1980; Theeuwes, 1992): salient stimuli capture attention involuntarily. We scale anomaly saliency from high (foreign language text) to ultra-low (single-character typos, off-by-one math errors).
- **Change blindness** (Rensink, O'Regan & Clark, 1997): failure to detect changes when attention is directed elsewhere — analogous to missing name inconsistencies or date errors embedded in routine prose.

### Difficulty Scaling

| Level    | Primary Task  | Anomaly Saliency | Passage Length | Anomaly Types                    |
|----------|--------------|------------------|----------------|----------------------------------|
| Easy     | count_word   | High             | 8 paragraphs   | language_switch, code_block       |
| Medium   | count_word   | High             | 15 paragraphs  | language_switch, code_block       |
| Hard     | count_names  | Medium           | 20 paragraphs  | factual_absurdity, numerical      |
| Expert   | sum_numbers  | Low              | 25 paragraphs  | name_inconsistency                |
| Frontier | sum_numbers  | Ultra-low        | 35 paragraphs  | single_char_swap, off_by_one, date|

### Scoring

SDK assertion pass rate. Two assertions per item: (1) primary answer within tolerance, (2) anomaly detection via keyword matching. Both must pass.

---

`<!-- COGATTENTION-BENCH-CANARY-7E5E99CE7F54 -->`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Imports + Inline Helpers
# CogAttention — Stimulus-Driven Attention
# ══════════════════════════════════════════════════════════════════════

import kaggle_benchmarks as kbench

import json
import re


def run_assertions_anomaly(response, gold, kbench):
    # The cognitive construct tested is anomaly detection, not exact counting.
    # The primary task (counting words/names/sums) is a routine cover task —
    # we only assert on whether the model detected the embedded anomaly.
    keywords = gold.get("detection_keywords", [])
    if keywords:
        alternatives = "|".join(re.escape(kw) for kw in keywords)
        pattern_anomaly = rf"(?i)({alternatives})"
        kbench.assertions.assert_contains_regex(
            pattern_anomaly, response,
            expectation=f"Should detect {gold['anomaly_type']} anomaly"
        )


print("CogAttention helpers loaded")
print(f"Task types: ['anomaly']")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Task Definitions + Embedded Dataset
# ══════════════════════════════════════════════════════════════════════


@kbench.task(name="cogattention_anomaly")
def cogattention_anomaly(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention anomaly task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_anomaly(response, gold, kbench)


# ── Embedded dataset ──────────────────────────────────────────────────
DATASET = json.loads(r'''
[
 {
  "task_id": "anomaly_easy_000",
  "task_type": "anomaly",
  "difficulty": "Easy",
  "prompt": "PRIMARY TASK: Count how many times the word \"was\" appears in the passage below.\n\n---\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nIl vecchio pescatore sedeva sul molo, osservando le onde con pazienza infinita.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n---\n\nAnswer BOTH questions:\n1. How many times does \"was\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 0, \"primary_task\": \"count_word\", \"anomaly_type\": \"language_switch\", \"detection_keywords\": [\"french\", \"spanish\", \"german\", \"italian\", \"foreign\", \"different language\", \"non-english\", \"another language\"]}"
 },
 {
  "task_id": "anomaly_easy_001",
  "task_type": "anomaly",
  "difficulty": "Easy",
  "prompt": "PRIMARY TASK: Count how many times the word \"was\" appears in the passage below.\n\n---\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nSELECT * FROM transactions WHERE amount > 1000 ORDER BY date DESC LIMIT 10;\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Freya fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n---\n\nAnswer BOTH questions:\n1. How many times does \"was\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 1, \"primary_task\": \"count_word\", \"anomaly_type\": \"code_block\", \"detection_keywords\": [\"code\", \"programming\", \"script\", \"function\", \"SQL\", \"software\", \"computer code\", \"snippet\"]}"
 },
 {
  "task_id": "anomaly_easy_002",
  "task_type": "anomaly",
  "difficulty": "Easy",
  "prompt": "PRIMARY TASK: Count how many times the word \"and\" appears in the passage below.\n\n---\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nDie alte Br\u00fccke \u00fcberquerte den Fluss seit mehr als dreihundert Jahren.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n---\n\nAnswer BOTH questions:\n1. How many times does \"and\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 2, \"primary_task\": \"count_word\", \"anomaly_type\": \"language_switch\", \"detection_keywords\": [\"french\", \"spanish\", \"german\", \"italian\", \"foreign\", \"different language\", \"non-english\", \"another language\"]}"
 },
 {
  "task_id": "anomaly_easy_003",
  "task_type": "anomaly",
  "difficulty": "Easy",
  "prompt": "PRIMARY TASK: Count how many times the word \"the\" appears in the passage below.\n\n---\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nLe soleil se couchait derri\u00e8re les montagnes, peignant le ciel en nuances d'or et de pourpre.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Priya Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n---\n\nAnswer BOTH questions:\n1. How many times does \"the\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 28, \"primary_task\": \"count_word\", \"anomaly_type\": \"language_switch\", \"detection_keywords\": [\"french\", \"spanish\", \"german\", \"italian\", \"foreign\", \"different language\", \"non-english\", \"another language\"]}"
 },
 {
  "task_id": "anomaly_easy_004",
  "task_type": "anomaly",
  "difficulty": "Easy",
  "prompt": "PRIMARY TASK: Count how many times the word \"and\" appears in the passage below.\n\n---\nA thin rain began to fall just as Joelle reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\ndef calculate_total(items): return sum(item.price for item in items if item.is_valid)\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Orla Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n---\n\nAnswer BOTH questions:\n1. How many times does \"and\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 5, \"primary_task\": \"count_word\", \"anomaly_type\": \"code_block\", \"detection_keywords\": [\"code\", \"programming\", \"script\", \"function\", \"SQL\", \"software\", \"computer code\", \"snippet\"]}"
 },
 {
  "task_id": "anomaly_easy_005",
  "task_type": "anomaly",
  "difficulty": "Easy",
  "prompt": "PRIMARY TASK: Count how many times the word \"and\" appears in the passage below.\n\n---\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nDie alte Br\u00fccke \u00fcberquerte den Fluss seit mehr als dreihundert Jahren.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Greta fully understood. The wooden shelves bowed slightly under the weight.\n---\n\nAnswer BOTH questions:\n1. How many times does \"and\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 3, \"primary_task\": \"count_word\", \"anomaly_type\": \"language_switch\", \"detection_keywords\": [\"french\", \"spanish\", \"german\", \"italian\", \"foreign\", \"different language\", \"non-english\", \"another language\"]}"
 },
 {
  "task_id": "anomaly_easy_006",
  "task_type": "anomaly",
  "difficulty": "Easy",
  "prompt": "PRIMARY TASK: Count how many times the word \"the\" appears in the passage below.\n\n---\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Lumi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\ndef calculate_total(items): return sum(item.price for item in items if item.is_valid)\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elio fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n---\n\nAnswer BOTH questions:\n1. How many times does \"the\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 24, \"primary_task\": \"count_word\", \"anomaly_type\": \"code_block\", \"detection_keywords\": [\"code\", \"programming\", \"script\", \"function\", \"SQL\", \"software\", \"computer code\", \"snippet\"]}"
 },
 {
  "task_id": "anomaly_easy_007",
  "task_type": "anomaly",
  "difficulty": "Easy",
  "prompt": "PRIMARY TASK: Count how many times the word \"the\" appears in the passage below.\n\n---\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nDie alte Br\u00fccke \u00fcberquerte den Fluss seit mehr als dreihundert Jahren.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Ines Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Bashir fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n---\n\nAnswer BOTH questions:\n1. How many times does \"the\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 26, \"primary_task\": \"count_word\", \"anomaly_type\": \"language_switch\", \"detection_keywords\": [\"french\", \"spanish\", \"german\", \"italian\", \"foreign\", \"different language\", \"non-english\", \"another language\"]}"
 },
 {
  "task_id": "anomaly_medium_008",
  "task_type": "anomaly",
  "difficulty": "Medium",
  "prompt": "PRIMARY TASK: Count how many times the word \"the\" appears in the passage below.\n\n---\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nDie alte Br\u00fccke \u00fcberquerte den Fluss seit mehr als dreihundert Jahren.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Olena fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elio fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n---\n\nAnswer BOTH questions:\n1. How many times does \"the\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 47, \"primary_task\": \"count_word\", \"anomaly_type\": \"language_switch\", \"detection_keywords\": [\"french\", \"spanish\", \"german\", \"italian\", \"foreign\", \"different language\", \"non-english\", \"another language\"]}"
 },
 {
  "task_id": "anomaly_medium_009",
  "task_type": "anomaly",
  "difficulty": "Medium",
  "prompt": "PRIMARY TASK: Count how many times the word \"that\" appears in the passage below.\n\n---\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Tala reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Gael reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nconst result = await fetch(\"/api/data\").then(res => res.json()).catch(err => console.error(err));\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Elara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient.\n---\n\nAnswer BOTH questions:\n1. How many times does \"that\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 3, \"primary_task\": \"count_word\", \"anomaly_type\": \"code_block\", \"detection_keywords\": [\"code\", \"programming\", \"script\", \"function\", \"SQL\", \"software\", \"computer code\", \"snippet\"]}"
 },
 {
  "task_id": "anomaly_medium_010",
  "task_type": "anomaly",
  "difficulty": "Medium",
  "prompt": "PRIMARY TASK: Count how many times the word \"was\" appears in the passage below.\n\n---\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Sigrid reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Vesna reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Soren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nfor (int i = 0; i < n; i++) { if (arr[i] > max) { max = arr[i]; } }\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n---\n\nAnswer BOTH questions:\n1. How many times does \"was\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 1, \"primary_task\": \"count_word\", \"anomaly_type\": \"code_block\", \"detection_keywords\": [\"code\", \"programming\", \"script\", \"function\", \"SQL\", \"software\", \"computer code\", \"snippet\"]}"
 },
 {
  "task_id": "anomaly_medium_011",
  "task_type": "anomaly",
  "difficulty": "Medium",
  "prompt": "PRIMARY TASK: Count how many times the word \"and\" appears in the passage below.\n\n---\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Tariq reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nDie alte Br\u00fccke \u00fcberquerte den Fluss seit mehr als dreihundert Jahren.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n---\n\nAnswer BOTH questions:\n1. How many times does \"and\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 6, \"primary_task\": \"count_word\", \"anomaly_type\": \"language_switch\", \"detection_keywords\": [\"french\", \"spanish\", \"german\", \"italian\", \"foreign\", \"different language\", \"non-english\", \"another language\"]}"
 },
 {
  "task_id": "anomaly_medium_012",
  "task_type": "anomaly",
  "difficulty": "Medium",
  "prompt": "PRIMARY TASK: Count how many times the word \"that\" appears in the passage below.\n\n---\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Adaeze Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Amara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nLe soleil se couchait derri\u00e8re les montagnes, peignant le ciel en nuances d'or et de pourpre.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n---\n\nAnswer BOTH questions:\n1. How many times does \"that\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 1, \"primary_task\": \"count_word\", \"anomaly_type\": \"language_switch\", \"detection_keywords\": [\"french\", \"spanish\", \"german\", \"italian\", \"foreign\", \"different language\", \"non-english\", \"another language\"]}"
 },
 {
  "task_id": "anomaly_medium_013",
  "task_type": "anomaly",
  "difficulty": "Medium",
  "prompt": "PRIMARY TASK: Count how many times the word \"and\" appears in the passage below.\n\n---\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Dariush reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\ndef calculate_total(items): return sum(item.price for item in items if item.is_valid)\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Tala reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Cusco was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Joelle Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n---\n\nAnswer BOTH questions:\n1. How many times does \"and\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 9, \"primary_task\": \"count_word\", \"anomaly_type\": \"code_block\", \"detection_keywords\": [\"code\", \"programming\", \"script\", \"function\", \"SQL\", \"software\", \"computer code\", \"snippet\"]}"
 },
 {
  "task_id": "anomaly_medium_014",
  "task_type": "anomaly",
  "difficulty": "Medium",
  "prompt": "PRIMARY TASK: Count how many times the word \"was\" appears in the passage below.\n\n---\nA thin rain began to fall just as Ines reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nSELECT * FROM transactions WHERE amount > 1000 ORDER BY date DESC LIMIT 10;\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Ugo Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Magnus fully understood. The wooden shelves bowed slightly under the weight.\n---\n\nAnswer BOTH questions:\n1. How many times does \"was\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 1, \"primary_task\": \"count_word\", \"anomaly_type\": \"code_block\", \"detection_keywords\": [\"code\", \"programming\", \"script\", \"function\", \"SQL\", \"software\", \"computer code\", \"snippet\"]}"
 },
 {
  "task_id": "anomaly_medium_015",
  "task_type": "anomaly",
  "difficulty": "Medium",
  "prompt": "PRIMARY TASK: Count how many times the word \"the\" appears in the passage below.\n\n---\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe workshop on Bram Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\nLe soleil se couchait derri\u00e8re les montagnes, peignant le ciel en nuances d'or et de pourpre.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nico fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Bram fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Tariq reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Xander reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Colette reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joaquin fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n---\n\nAnswer BOTH questions:\n1. How many times does \"the\" appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 46, \"primary_task\": \"count_word\", \"anomaly_type\": \"language_switch\", \"detection_keywords\": [\"french\", \"spanish\", \"german\", \"italian\", \"foreign\", \"different language\", \"non-english\", \"another language\"]}"
 },
 {
  "task_id": "anomaly_hard_016",
  "task_type": "anomaly",
  "difficulty": "Hard",
  "prompt": "PRIMARY TASK: Count how many different person names are mentioned in the passage below.\n\n---\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Bram fully understood. The wooden shelves bowed slightly under the weight.\n\nThe Atlantic Ocean, the smallest of Earth's oceans, separates Europe from the Americas.\n\nThe workshop on Lumi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dmitri fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Oulu insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Elio reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Joaquin Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n---\n\nAnswer BOTH questions:\n1. How many different person names appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 5, \"primary_task\": \"count_names\", \"anomaly_type\": \"factual_absurdity\", \"detection_keywords\": [\"incorrect\", \"wrong\", \"false\", \"absurd\", \"impossible\", \"factual error\", \"inaccurate\", \"not true\", \"east\", \"uphill\", \"smallest\", \"forty-five\"]}"
 },
 {
  "task_id": "anomaly_hard_017",
  "task_type": "anomaly",
  "difficulty": "Hard",
  "prompt": "PRIMARY TASK: Count how many different person names are mentioned in the passage below.\n\n---\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Adaeze fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe temperature that January reached forty-five degrees Celsius in Helsinki, a typical winter day.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Elio Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe workshop on Femi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Yuki reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n---\n\nAnswer BOTH questions:\n1. How many different person names appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 5, \"primary_task\": \"count_names\", \"anomaly_type\": \"factual_absurdity\", \"detection_keywords\": [\"incorrect\", \"wrong\", \"false\", \"absurd\", \"impossible\", \"factual error\", \"inaccurate\", \"not true\", \"east\", \"uphill\", \"smallest\", \"forty-five\"]}"
 },
 {
  "task_id": "anomaly_hard_018",
  "task_type": "anomaly",
  "difficulty": "Hard",
  "prompt": "PRIMARY TASK: Count how many different person names are mentioned in the passage below.\n\n---\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Joaquin reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Lumi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe workshop on Viktor Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe final invoice for a single cup of coffee came to $12,000,000.00, which Priya paid without hesitation.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Soren fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Idris Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Wren fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n---\n\nAnswer BOTH questions:\n1. How many different person names appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 8, \"primary_task\": \"count_names\", \"anomaly_type\": \"numerical_anomaly\", \"detection_keywords\": [\"large\", \"unusual\", \"amount\", \"million\", \"expensive\", \"anomalous\", \"extraordinary\", \"999\", \"8,500\", \"12,000\", \"unrealistic\", \"absurd\", \"strange\", \"odd\", \"suspicious\", \"outrageous\", \"invoice\", \"exorbitant\", \"ridiculous\", \"implausible\", \"enormous\", \"huge\", \"unreasonable\", \"clearly wrong\", \"fabricated\", \"not realistic\", \"absurdly\", \"coffee\", \"overpriced\", \"inflated\", \"astronomical\", \"excessive\", \"disproportionate\", \"anomaly\", \"surprising\", \"noteworthy\", \"questionable\", \"peculiar\", \"bizarre\", \"preposterous\"]}"
 },
 {
  "task_id": "anomaly_hard_019",
  "task_type": "anomaly",
  "difficulty": "Hard",
  "prompt": "PRIMARY TASK: Count how many different person names are mentioned in the passage below.\n\n---\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nalini fully understood. The wooden shelves bowed slightly under the weight.\n\nThe final invoice for a single cup of coffee came to $8,500,000.00, which Sigrid paid without hesitation.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Zain Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe workshop on Lumi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Elio reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n---\n\nAnswer BOTH questions:\n1. How many different person names appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 6, \"primary_task\": \"count_names\", \"anomaly_type\": \"numerical_anomaly\", \"detection_keywords\": [\"large\", \"unusual\", \"amount\", \"million\", \"expensive\", \"anomalous\", \"extraordinary\", \"999\", \"8,500\", \"12,000\", \"unrealistic\", \"absurd\", \"strange\", \"odd\", \"suspicious\", \"outrageous\", \"invoice\", \"exorbitant\", \"ridiculous\", \"implausible\", \"enormous\", \"huge\", \"unreasonable\", \"clearly wrong\", \"fabricated\", \"not realistic\", \"absurdly\", \"coffee\", \"overpriced\", \"inflated\", \"astronomical\", \"excessive\", \"disproportionate\", \"anomaly\", \"surprising\", \"noteworthy\", \"questionable\", \"peculiar\", \"bizarre\", \"preposterous\"]}"
 },
 {
  "task_id": "anomaly_hard_020",
  "task_type": "anomaly",
  "difficulty": "Hard",
  "prompt": "PRIMARY TASK: Count how many different person names are mentioned in the passage below.\n\n---\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Greta fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Yuki Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tala fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Nico reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe final invoice for a single cup of coffee came to $999,999.00, which Dmitri paid without hesitation.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA thin rain began to fall just as Uma reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n---\n\nAnswer BOTH questions:\n1. How many different person names appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 7, \"primary_task\": \"count_names\", \"anomaly_type\": \"numerical_anomaly\", \"detection_keywords\": [\"large\", \"unusual\", \"amount\", \"million\", \"expensive\", \"anomalous\", \"extraordinary\", \"999\", \"8,500\", \"12,000\", \"unrealistic\", \"absurd\", \"strange\", \"odd\", \"suspicious\", \"outrageous\", \"invoice\", \"exorbitant\", \"ridiculous\", \"implausible\", \"enormous\", \"huge\", \"unreasonable\", \"clearly wrong\", \"fabricated\", \"not realistic\", \"absurdly\", \"coffee\", \"overpriced\", \"inflated\", \"astronomical\", \"excessive\", \"disproportionate\", \"anomaly\", \"surprising\", \"noteworthy\", \"questionable\", \"peculiar\", \"bizarre\", \"preposterous\"]}"
 },
 {
  "task_id": "anomaly_hard_021",
  "task_type": "anomaly",
  "difficulty": "Hard",
  "prompt": "PRIMARY TASK: Count how many different person names are mentioned in the passage below.\n\n---\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Maren fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Amara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Dariush reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Freya fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe final invoice for a single cup of coffee came to $12,000,000.00, which Qadir paid without hesitation.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dariush fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n---\n\nAnswer BOTH questions:\n1. How many different person names appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 6, \"primary_task\": \"count_names\", \"anomaly_type\": \"numerical_anomaly\", \"detection_keywords\": [\"large\", \"unusual\", \"amount\", \"million\", \"expensive\", \"anomalous\", \"extraordinary\", \"999\", \"8,500\", \"12,000\", \"unrealistic\", \"absurd\", \"strange\", \"odd\", \"suspicious\", \"outrageous\", \"invoice\", \"exorbitant\", \"ridiculous\", \"implausible\", \"enormous\", \"huge\", \"unreasonable\", \"clearly wrong\", \"fabricated\", \"not realistic\", \"absurdly\", \"coffee\", \"overpriced\", \"inflated\", \"astronomical\", \"excessive\", \"disproportionate\", \"anomaly\", \"surprising\", \"noteworthy\", \"questionable\", \"peculiar\", \"bizarre\", \"preposterous\"]}"
 },
 {
  "task_id": "anomaly_hard_022",
  "task_type": "anomaly",
  "difficulty": "Hard",
  "prompt": "PRIMARY TASK: Count how many different person names are mentioned in the passage below.\n\n---\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Cusco was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Ines reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Nico reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nico fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nWater, as everyone knows, flows uphill when left undisturbed in open channels.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Femi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Uma fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dmitri fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n---\n\nAnswer BOTH questions:\n1. How many different person names appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 6, \"primary_task\": \"count_names\", \"anomaly_type\": \"factual_absurdity\", \"detection_keywords\": [\"incorrect\", \"wrong\", \"false\", \"absurd\", \"impossible\", \"factual error\", \"inaccurate\", \"not true\", \"east\", \"uphill\", \"smallest\", \"forty-five\"]}"
 },
 {
  "task_id": "anomaly_hard_023",
  "task_type": "anomaly",
  "difficulty": "Hard",
  "prompt": "PRIMARY TASK: Count how many different person names are mentioned in the passage below.\n\n---\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe final invoice for a single cup of coffee came to $12,000,000.00, which Femi paid without hesitation.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Femi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Orla Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Kaia Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joelle fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Vesna reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n---\n\nAnswer BOTH questions:\n1. How many different person names appear?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 6, \"primary_task\": \"count_names\", \"anomaly_type\": \"numerical_anomaly\", \"detection_keywords\": [\"large\", \"unusual\", \"amount\", \"million\", \"expensive\", \"anomalous\", \"extraordinary\", \"999\", \"8,500\", \"12,000\", \"unrealistic\", \"absurd\", \"strange\", \"odd\", \"suspicious\", \"outrageous\", \"invoice\", \"exorbitant\", \"ridiculous\", \"implausible\", \"enormous\", \"huge\", \"unreasonable\", \"clearly wrong\", \"fabricated\", \"not realistic\", \"absurdly\", \"coffee\", \"overpriced\", \"inflated\", \"astronomical\", \"excessive\", \"disproportionate\", \"anomaly\", \"surprising\", \"noteworthy\", \"questionable\", \"peculiar\", \"bizarre\", \"preposterous\"]}"
 },
 {
  "task_id": "anomaly_expert_024",
  "task_type": "anomaly",
  "difficulty": "Expert",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nThe workshop on Tariq Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The cost came to $112.34.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The cost came to $168.57.\n\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The cost came to $47.03.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The cost came to $157.31.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nAmara checked the inventory one final time. Later that afternoon, Viktor \u2014 who is actually the same person as Amara, now inexplicably referred to by a completely different name \u2014 signed off on the delivery manifest.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Amara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zain fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Dmitri Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Lumi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Qadir fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Sigrid fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 485.25, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"name_inconsistency\", \"detection_keywords\": [\"name\", \"changed\", \"inconsistent\", \"different name\", \"was called\", \"previously\", \"earlier\", \"same person\", \"same individual\", \"two different\", \"refers to\", \"contradiction\", \"conflicting\", \"switched\", \"mixed up\", \"confused\", \"swapped\", \"identity\", \"referred\", \"called\", \"mismatch\", \"discrepancy\", \"oddly\", \"another name\", \"renamed\", \"alias\", \"unclear\", \"who\"]}"
 },
 {
  "task_id": "anomaly_expert_025",
  "task_type": "anomaly",
  "difficulty": "Expert",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe workshop on Leif Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nBashir checked the inventory one final time. Later that afternoon, Kaia \u2014 who is actually the same person as Bashir, now inexplicably referred to by a completely different name \u2014 signed off on the delivery manifest.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yara fully understood. The wooden shelves bowed slightly under the weight. The cost came to $72.88.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dariush fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The cost came to $62.94.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe workshop on Vesna Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Willa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The cost came to $109.05.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient. The cost came to $122.86.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Xander fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Soren fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Zora Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 367.73, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"name_inconsistency\", \"detection_keywords\": [\"name\", \"changed\", \"inconsistent\", \"different name\", \"was called\", \"previously\", \"earlier\", \"same person\", \"same individual\", \"two different\", \"refers to\", \"contradiction\", \"conflicting\", \"switched\", \"mixed up\", \"confused\", \"swapped\", \"identity\", \"referred\", \"called\", \"mismatch\", \"discrepancy\", \"oddly\", \"another name\", \"renamed\", \"alias\", \"unclear\", \"who\"]}"
 },
 {
  "task_id": "anomaly_expert_026",
  "task_type": "anomaly",
  "difficulty": "Expert",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Qadir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The cost came to $24.37.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient.\n\nMagnus checked the inventory one final time. Later that afternoon, Qadir \u2014 who is actually the same person as Magnus, now inexplicably referred to by a completely different name \u2014 signed off on the delivery manifest.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Yuki reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The cost came to $134.73.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The cost came to $195.88.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Cusco was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Xander reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Yuki Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The cost came to $171.64.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Tariq Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Femi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Haruto Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 526.62, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"name_inconsistency\", \"detection_keywords\": [\"name\", \"changed\", \"inconsistent\", \"different name\", \"was called\", \"previously\", \"earlier\", \"same person\", \"same individual\", \"two different\", \"refers to\", \"contradiction\", \"conflicting\", \"switched\", \"mixed up\", \"confused\", \"swapped\", \"identity\", \"referred\", \"called\", \"mismatch\", \"discrepancy\", \"oddly\", \"another name\", \"renamed\", \"alias\", \"unclear\", \"who\"]}"
 },
 {
  "task_id": "anomaly_expert_027",
  "task_type": "anomaly",
  "difficulty": "Expert",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nThe workshop on Bram Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Uma Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The cost came to $115.60.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Vesna fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Greta reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Yuki reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The cost came to $167.40.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Gael Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe workshop on Yuki Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nPriya checked the inventory one final time. Later that afternoon, Joaquin \u2014 who is actually the same person as Priya, now inexplicably referred to by a completely different name \u2014 signed off on the delivery manifest.\n\nA thin rain began to fall just as Bashir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The cost came to $68.10.\n\nA thin rain began to fall just as Uma reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Kaia fully understood. The wooden shelves bowed slightly under the weight. The cost came to $155.23.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 506.33, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"name_inconsistency\", \"detection_keywords\": [\"name\", \"changed\", \"inconsistent\", \"different name\", \"was called\", \"previously\", \"earlier\", \"same person\", \"same individual\", \"two different\", \"refers to\", \"contradiction\", \"conflicting\", \"switched\", \"mixed up\", \"confused\", \"swapped\", \"identity\", \"referred\", \"called\", \"mismatch\", \"discrepancy\", \"oddly\", \"another name\", \"renamed\", \"alias\", \"unclear\", \"who\"]}"
 },
 {
  "task_id": "anomaly_expert_028",
  "task_type": "anomaly",
  "difficulty": "Expert",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The cost came to $72.93.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. The cost came to $92.50.\n\nYuki checked the inventory one final time. Later that afternoon, Leif \u2014 who is actually the same person as Yuki, now inexplicably referred to by a completely different name \u2014 signed off on the delivery manifest.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe workshop on Nalini Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe workshop on Xander Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The cost came to $180.25.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient. The cost came to $140.12.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tala fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 485.8, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"name_inconsistency\", \"detection_keywords\": [\"name\", \"changed\", \"inconsistent\", \"different name\", \"was called\", \"previously\", \"earlier\", \"same person\", \"same individual\", \"two different\", \"refers to\", \"contradiction\", \"conflicting\", \"switched\", \"mixed up\", \"confused\", \"swapped\", \"identity\", \"referred\", \"called\", \"mismatch\", \"discrepancy\", \"oddly\", \"another name\", \"renamed\", \"alias\", \"unclear\", \"who\"]}"
 },
 {
  "task_id": "anomaly_expert_029",
  "task_type": "anomaly",
  "difficulty": "Expert",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The cost came to $46.87.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Paloma fully understood. The wooden shelves bowed slightly under the weight. The cost came to $118.06.\n\nThe workshop on Tala Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Qadir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient. The cost came to $60.28.\n\nNico checked the inventory one final time. Later that afternoon, Gael \u2014 who is actually the same person as Nico, now inexplicably referred to by a completely different name \u2014 signed off on the delivery manifest.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The cost came to $95.57.\n\nThe workshop on Uma Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Kaia Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Xander reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 320.78, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"name_inconsistency\", \"detection_keywords\": [\"name\", \"changed\", \"inconsistent\", \"different name\", \"was called\", \"previously\", \"earlier\", \"same person\", \"same individual\", \"two different\", \"refers to\", \"contradiction\", \"conflicting\", \"switched\", \"mixed up\", \"confused\", \"swapped\", \"identity\", \"referred\", \"called\", \"mismatch\", \"discrepancy\", \"oddly\", \"another name\", \"renamed\", \"alias\", \"unclear\", \"who\"]}"
 },
 {
  "task_id": "anomaly_expert_030",
  "task_type": "anomaly",
  "difficulty": "Expert",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Gael reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The cost came to $53.22.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Willa fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe workshop on Ravi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zain fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The cost came to $187.17.\n\nThe workshop on Magnus Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Lumi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nico fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. The cost came to $12.20.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The cost came to $126.15.\n\nQadir checked the inventory one final time. Later that afternoon, Femi \u2014 who is actually the same person as Qadir, now inexplicably referred to by a completely different name \u2014 signed off on the delivery manifest.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Oulu insisted on maintaining their position, while the merchants grew impatient.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 378.74, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"name_inconsistency\", \"detection_keywords\": [\"name\", \"changed\", \"inconsistent\", \"different name\", \"was called\", \"previously\", \"earlier\", \"same person\", \"same individual\", \"two different\", \"refers to\", \"contradiction\", \"conflicting\", \"switched\", \"mixed up\", \"confused\", \"swapped\", \"identity\", \"referred\", \"called\", \"mismatch\", \"discrepancy\", \"oddly\", \"another name\", \"renamed\", \"alias\", \"unclear\", \"who\"]}"
 },
 {
  "task_id": "anomaly_expert_031",
  "task_type": "anomaly",
  "difficulty": "Expert",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The cost came to $175.29.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Adaeze reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nInes checked the inventory one final time. Later that afternoon, Joelle \u2014 who is actually the same person as Ines, now inexplicably referred to by a completely different name \u2014 signed off on the delivery manifest.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The cost came to $182.84.\n\nThe workshop on Zain Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Freya fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elio fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Ines Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe workshop on Leif Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The cost came to $81.10.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Lumi fully understood. The wooden shelves bowed slightly under the weight. The cost came to $193.35.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe workshop on Tala Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 632.58, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"name_inconsistency\", \"detection_keywords\": [\"name\", \"changed\", \"inconsistent\", \"different name\", \"was called\", \"previously\", \"earlier\", \"same person\", \"same individual\", \"two different\", \"refers to\", \"contradiction\", \"conflicting\", \"switched\", \"mixed up\", \"confused\", \"swapped\", \"identity\", \"referred\", \"called\", \"mismatch\", \"discrepancy\", \"oddly\", \"another name\", \"renamed\", \"alias\", \"unclear\", \"who\"]}"
 },
 {
  "task_id": "anomaly_frontier_032",
  "task_type": "anomaly",
  "difficulty": "Frontier",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Celine Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The cost came to $118.97.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Viktor fully understood. The wooden shelves bowed slightly under the weight. The cost came to $149.95.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The cost came to $74.26.\n\nThe workshop on Joelle Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Hana reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The cost came to $65.50.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Vesna reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Maren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Orla Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe bridge, completed in 1889, celebrated its 150th anniversary in 2038.\n\nA thin rain began to fall just as Wren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Kaia reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Elara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 408.68, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"off_by_one\", \"detection_keywords\": [\"math\", \"calculation\", \"wrong\", \"incorrect\", \"off by one\", \"doesn't add up\", \"inconsistent\", \"150\", \"anniversary\", \"groups of 5\", \"12\", \"48-hour\", \"Wednesday\", \"8 eggs\", \"7\", \"error\", \"mistake\", \"mismatch\", \"doesn't match\", \"not correct\", \"arithmetic\", \"discrepancy\", \"inaccurate\", \"impossible\", \"illogical\", \"contradicts\", \"should be\", \"actually\", \"149\", \"1889\", \"2038\", \"Monday\", \"expired\", \"deadline\", \"split\", \"recipe\", \"eggs\"]}"
 },
 {
  "task_id": "anomaly_frontier_033",
  "task_type": "anomaly",
  "difficulty": "Frontier",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nA thin rain began to fall just as Lumi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The cost came to $96.10.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The cost came to $191.68.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dmitri fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elio fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Yuki reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Runa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nAll shipments were recieved in good condition according to the warehouse manifest.\n\nThe workshop on Orla Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Ugo Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Tariq reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Qadir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Joaquin Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The cost came to $16.97.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient. The cost came to $33.96.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Nico reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 338.71, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"single_char_swap\", \"detection_keywords\": [\"typo\", \"misspelling\", \"spelling\", \"buidling\", \"reveneu\", \"recieved\", \"committe\", \"error\", \"mistake\"]}"
 },
 {
  "task_id": "anomaly_frontier_034",
  "task_type": "anomaly",
  "difficulty": "Frontier",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Bashir fully understood. The wooden shelves bowed slightly under the weight. The cost came to $41.21.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Zora Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Bashir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nalini fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nThe meeting on Wednesday, April 22nd was rescheduled to Thursday, April 22nd.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient. The cost came to $106.95.\n\nA thin rain began to fall just as Ugo reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Lumi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Zain Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The cost came to $60.90.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Magnus reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Viktor Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Willa fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zora fully understood. The wooden shelves bowed slightly under the weight. The cost came to $50.49.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Vesna fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 259.55, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"date_inconsistency\", \"detection_keywords\": [\"date\", \"calendar\", \"day\", \"wrong\", \"inconsistent\", \"February 29\", \"doesn't match\", \"Tuesday\", \"March 15\", \"Saturday\", \"June 7\", \"error\", \"mistake\", \"mismatch\", \"incorrect\", \"not correct\", \"discrepancy\", \"impossible\"]}"
 },
 {
  "task_id": "anomaly_frontier_035",
  "task_type": "anomaly",
  "difficulty": "Frontier",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nThe workshop on Ugo Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Paloma Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe workshop on Haruto Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Lumi fully understood. The wooden shelves bowed slightly under the weight. The cost came to $118.90.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yara fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient. The cost came to $95.39.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Haruto reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Paloma reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The cost came to $79.96.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nico fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Greta reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Soren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The cost came to $64.28.\n\nThe 48-hour deadline began on Monday and expired on Wednesday at the same hour.\n\nThe workshop on Femi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Xander fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Elara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Ines reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 358.53, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"off_by_one\", \"detection_keywords\": [\"math\", \"calculation\", \"wrong\", \"incorrect\", \"off by one\", \"doesn't add up\", \"inconsistent\", \"150\", \"anniversary\", \"groups of 5\", \"12\", \"48-hour\", \"Wednesday\", \"8 eggs\", \"7\", \"error\", \"mistake\", \"mismatch\", \"doesn't match\", \"not correct\", \"arithmetic\", \"discrepancy\", \"inaccurate\", \"impossible\", \"illogical\", \"contradicts\", \"should be\", \"actually\", \"149\", \"1889\", \"2038\", \"Monday\", \"expired\", \"deadline\", \"split\", \"recipe\", \"eggs\"]}"
 },
 {
  "task_id": "anomaly_frontier_036",
  "task_type": "anomaly",
  "difficulty": "Frontier",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The cost came to $110.36.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe workshop on Uma Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Qadir fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Elara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dariush fully understood. The wooden shelves bowed slightly under the weight. The cost came to $14.18.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Xander Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA thin rain began to fall just as Kaia reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The cost came to $123.08.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe shipment departed on February 29th, 2023 and arrived three days later.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight. The cost came to $35.58.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Haruto Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Hana reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Wren fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Uma reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 283.2, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"date_inconsistency\", \"detection_keywords\": [\"date\", \"calendar\", \"day\", \"wrong\", \"inconsistent\", \"February 29\", \"doesn't match\", \"Tuesday\", \"March 15\", \"Saturday\", \"June 7\", \"error\", \"mistake\", \"mismatch\", \"incorrect\", \"not correct\", \"discrepancy\", \"impossible\"]}"
 },
 {
  "task_id": "anomaly_frontier_037",
  "task_type": "anomaly",
  "difficulty": "Frontier",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Elio reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient. The cost came to $35.09.\n\nThe workshop on Zain Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Ugo reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Joaquin Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joelle fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The cost came to $196.01.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Sigrid fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Colette Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Greta reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Lumi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Lumi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe bridge, completed in 1889, celebrated its 150th anniversary in 2038.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The cost came to $176.36.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zora fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient. The cost came to $147.35.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 554.81, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"off_by_one\", \"detection_keywords\": [\"math\", \"calculation\", \"wrong\", \"incorrect\", \"off by one\", \"doesn't add up\", \"inconsistent\", \"150\", \"anniversary\", \"groups of 5\", \"12\", \"48-hour\", \"Wednesday\", \"8 eggs\", \"7\", \"error\", \"mistake\", \"mismatch\", \"doesn't match\", \"not correct\", \"arithmetic\", \"discrepancy\", \"inaccurate\", \"impossible\", \"illogical\", \"contradicts\", \"should be\", \"actually\", \"149\", \"1889\", \"2038\", \"Monday\", \"expired\", \"deadline\", \"split\", \"recipe\", \"eggs\"]}"
 },
 {
  "task_id": "anomaly_frontier_038",
  "task_type": "anomaly",
  "difficulty": "Frontier",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The cost came to $166.21.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Paloma fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The cost came to $93.95.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe workshop on Ines Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Lumi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe workshop on Leif Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Elio reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The cost came to $111.04.\n\nA thin rain began to fall just as Ines reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tala fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Vesna fully understood. The wooden shelves bowed slightly under the weight.\n\nThe committe approved the proposal after reviewing the supplementary documentation.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Femi fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Tariq reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Vesna reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Dariush Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Adaeze reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The cost came to $54.92.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Magnus fully understood. The wooden shelves bowed slightly under the weight.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 426.12, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"single_char_swap\", \"detection_keywords\": [\"typo\", \"misspelling\", \"spelling\", \"buidling\", \"reveneu\", \"recieved\", \"committe\", \"error\", \"mistake\"]}"
 },
 {
  "task_id": "anomaly_frontier_039",
  "task_type": "anomaly",
  "difficulty": "Frontier",
  "prompt": "PRIMARY TASK: Find all the dollar amounts mentioned in the passage and compute their sum.\n\n---\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Maren fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Willa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The cost came to $166.79.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Olena fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Nico reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Lumi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Cusco was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The cost came to $91.57.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The cost came to $95.63.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Uma Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Nalini Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe inspection revealed that the buidling met all safety standards without exception.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Elio Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The cost came to $119.45.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n---\n\nAnswer BOTH questions:\n1. What is the total sum of all dollar amounts?\n2. Did you notice anything unusual or out of place in the passage? Look for things like: text in a foreign language, code/programming syntax, factual errors, impossible numbers, spelling mistakes, or date/math errors. Ignore any repetitive sentence structures \u2014 focus on content anomalies only. If yes, describe it briefly. If no, write 'nothing unusual'.\n\nFormat:\nANSWER:\n1. [your answer]\n2. [description or 'nothing unusual']",
  "gold_json": "{\"primary_answer\": 473.44, \"primary_task\": \"sum_numbers\", \"anomaly_type\": \"single_char_swap\", \"detection_keywords\": [\"typo\", \"misspelling\", \"spelling\", \"buidling\", \"reveneu\", \"recieved\", \"committe\", \"error\", \"mistake\"]}"
 }
]
''')

print(f"Loaded {len(DATASET)} items")
for tt in ['anomaly']:
    count = sum(1 for d in DATASET if d["task_type"] == tt)
    print(f"  {tt}: {count} items")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Execution Loop
# ══════════════════════════════════════════════════════════════════════

TASK_DISPATCH = {
    "anomaly": cogattention_anomaly,
}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Stimulus-Driven Attention")
